# Interpolation Feature Comparison

Compare the template interpolation methods used in the paper:
- **Unprocessed** (measured HRTFs)
- **barumerli2023** (SH order 15 + Tikhonov)
- **SH** (auto-order spherical harmonics)
- **SHMAX** (order 44 + Tikhonov + Bau damping)
- **barycentric** (VBAP-based)

This notebook computes the template reconstruction error (RMSE of the spectral cues) for the 33
participants reported in Sec. 3.2 and writes `results/interpolation_errors.csv`.
It needs the participant HRTFs (`data/hrtf/`, downloaded by `init.sh`) and the behavioural data
(`data/behavioural/AXD_full_dataset_20260202.csv`, for the participant list).

The figures based on the KEMAR HRTF are produced by the figure scripts:
Fig. 1 `figures/fig_posterior_distribution.py`, Fig. 2 `figures/fig_grid_visualization.py`,
Fig. 3 and Fig. S1 `figures/fig_sh_methods.py`.

**Note.** `results/interpolation_errors.csv` holds the values reported in the paper, computed with
the development version of `bayesian_listener` used for the paper (commit `c0b711a`). The released
package computes the spectral features much faster, at a different absolute level. As in the code
used for the paper, `barumerli2023` regularises the order-0 SH coefficient (`SIG[1:9, 1:9] = 0`),
whereas the original MATLAB implementation (AMT) leaves orders 0–2 unregularised
(`SIG(1:9, 1:9) = 0`, 1-based), so its template error depends on the feature level: re-running this
notebook gives slightly different values for barumerli2023 and overwrites the CSV.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import pyfar as pf

from bayesian_listener import BayesianListener

import paths

plt.rcParams.update({'font.size': 10})

## 1. Load KEMAR and prepare features for all interpolation methods

In [2]:
sofa_path = str(paths.KEMAR_SOFA)  # data/hrtf/KEMAR_FreeFieldCompMinPhase_48kHz.sofa

methods = ['barumerli2023', 'SH', 'SHMAX', 'barycentric']
models = {}

for method in methods:
    print(f'Preparing {method}...')
    m = BayesianListener(sofa_path)
    m.compute_template(interpolation=method, cache_dir=paths.CACHE_DIR)
    models[method] = m
    print(f'  Original: {m.target.coords.spherical_elevation.shape[0]} directions')
    print(f'  Template: {m.template.coords.spherical_elevation.shape[0]} directions')

# Use the last model's unprocessed features as reference
# (all models share the same unprocessed features)
ref = models['SHMAX']
print(f'\nFrequencies: {len(ref.template.freqs)} bands, {ref.template.freqs[0]:.0f} - {ref.template.freqs[-1]:.0f} Hz')

# Check available polar angles
hpo_ref = np.rad2deg(ref.target.coords.spherical_side)
print(f'\nUnprocessed polar angle range: {hpo_ref[:, 1].min():.1f} to {hpo_ref[:, 1].max():.1f} deg')
print(f'Unprocessed lateral angle range: {hpo_ref[:, 0].min():.1f} to {hpo_ref[:, 0].max():.1f} deg')

Preparing barumerli2023...
  Original: 793 directions
  Template: 2112 directions
Preparing SH...
  Original: 793 directions
  Template: 2112 directions
Preparing SHMAX...
  Original: 793 directions
  Template: 2112 directions
Preparing barycentric...
  Original: 793 directions
  Template: 2112 directions

Frequencies: 28 bands, 729 - 17439 Hz

Unprocessed polar angle range: -90.0 to 270.0 deg
Unprocessed lateral angle range: -90.0 to 90.0 deg


## 5. Interpolation Error Analysis (All Participants)

Compute RMSE and Standard Error for each interpolation method across all participants used in the fitting study. For each participant and method, we compare the original HRTF spectral features with the interpolated features at the nearest template points.

In [3]:
# Load participant list from the behavioural data
obs_tbl = pd.read_csv(paths.BEHAV_FILE)
obs_tbl = obs_tbl[(obs_tbl['condition'] == 'Measured')]
obs_tbl = obs_tbl[(obs_tbl['participant'] != 'P0247')]
subjects = obs_tbl['participant'].unique()

print(f'Found {len(subjects)} participants')
print(f'Methods to test: {methods}')

def compute_interpolation_errors(original_model, interpolated_model):
    """
    Compute RMSE and SE for spectral features between original and interpolated HRTFs.

    For each direction in the original HRTF, find the nearest point in the template
    and compare spectral features.

    Returns:
        rmse_left: RMSE for left ear
        rmse_right: RMSE for right ear
        se_left: Standard error for left ear
        se_right: Standard error for right ear
    """
    orig_coords = original_model.target.coords
    template_coords = interpolated_model.template.coords

    # For each original position, find nearest template position
    nearest_indices = template_coords.find_nearest(orig_coords)[0][0]

    # Get spectral features
    orig_spectral = original_model.target.spectral_cues  # (n_orig, n_freqs, 2)
    interp_spectral = interpolated_model.template.spectral_cues[nearest_indices]  # (n_orig, n_freqs, 2)

    # Compute errors for each ear
    errors_left = orig_spectral[:, :, 0] - interp_spectral[:, :, 0]  # (n_orig, n_freqs)
    errors_right = orig_spectral[:, :, 1] - interp_spectral[:, :, 1]

    # RMSE (root mean squared error)
    rmse_left = np.sqrt(np.mean(errors_left**2))
    rmse_right = np.sqrt(np.mean(errors_right**2))

    # Standard error (SE = std / sqrt(n))
    n_samples = errors_left.size
    se_left = np.std(errors_left) / np.sqrt(n_samples)
    se_right = np.std(errors_right) / np.sqrt(n_samples)

    return rmse_left, rmse_right, se_left, se_right


# Compute errors for all subjects and methods
all_error_results = []

for subj_idx, subj in enumerate(subjects):
    sofa_path = str(paths.participant_sofa(subj))
    print(f'[{subj_idx+1}/{len(subjects)}] Processing {subj}...', end=' ')

    try:
        # Load models for this subject
        subj_models = {}
        for method in methods:
            model = BayesianListener(sofa_path)
            model.compute_template(interpolation=method, cache_dir=paths.CACHE_DIR)
            subj_models[method] = model

        # Use one model as reference (they all share the same unprocessed features)
        ref_model = subj_models[methods[0]]

        # Compute errors for each method
        for method in methods:
            rmse_l, rmse_r, se_l, se_r = compute_interpolation_errors(ref_model, subj_models[method])
            all_error_results.append({
                'subject': subj,
                'method': method,
                'RMSE_left': rmse_l,
                'RMSE_right': rmse_r,
                'SE_left': se_l,
                'SE_right': se_r,
                'RMSE_mean': (rmse_l + rmse_r) / 2,
            })

        print('✓')

    except Exception as e:
        print(f'✗ Error: {e}')
        continue

# Create DataFrame
all_errors_df = pd.DataFrame(all_error_results)

print(f'\nSuccessfully processed {len(all_errors_df["subject"].unique())} subjects')
print(f'Total entries: {len(all_errors_df)}')

Found 33 participants
Methods to test: ['barumerli2023', 'SH', 'SHMAX', 'barycentric']
[1/33] Processing P0001... ✓
[2/33] Processing P0006... 

✓
[3/33] Processing P0019... 

✓
[4/33] Processing P0031... ✓
[5/33] Processing P0043... 

✓
[6/33] Processing P0124... 

✓
[7/33] Processing P0172... ✓
[8/33] Processing P0181... 

...

✓
[28/33] Processing P0330... ✓
[29/33] Processing P0347... 

✓
[30/33] Processing P0348... 

✓
[31/33] Processing P0349... ✓
[32/33] Processing P0350... 

✓
[33/33] Processing P0351... 

✓

Successfully processed 33 subjects
Total entries: 132


## 6. Grid sizes

Size of the measurement grid, of the complemented bottom cap and of the template grid
(visualised in Fig. 2 by `figures/fig_grid_visualization.py`).

In [4]:
from bayesian_listener.resample import complement_sampling

# Get coordinates from one of the models (they all share the same structure)
example_model = models['SHMAX']

# Original HRTF grid — coords is already pyfar.Coordinates
orig_coords = example_model.target.coords

orig_sph = np.rad2deg(orig_coords.spherical_elevation)  # (az_deg, el_deg, r)
orig_az_deg = orig_sph[:, 0]
orig_el_deg = orig_sph[:, 1]

# At the pole (el=90°) azimuth is undefined: keep only az=0° to avoid clutter
pole_mask = (orig_el_deg >= 89.9) & (np.abs(orig_az_deg) > 0.01)
orig_az_deg = orig_az_deg[~pole_mask]
orig_el_deg = orig_el_deg[~pole_mask]
# pyfar returns azimuth in [0, 360]; wrap to [-180, 180]
orig_az_deg = (orig_az_deg + 180) % 360 - 180

# Template grid (t-design 2112-point target)
template_sph = np.rad2deg(example_model.template.coords.spherical_elevation)
template_az_deg = template_sph[:, 0]
template_el_deg = template_sph[:, 1]
# wrap to [-180, 180]
template_az_deg = (template_az_deg + 180) % 360 - 180

# Complemented source grid: same call as in resample_two_step
# orig_coords is already pyfar, pass directly
coords_complemented, complement_mask = complement_sampling(orig_coords)

# Read azimuth/elevation directly from pyfar's spherical_elevation (az unchanged by mirroring)
comp_sph_el = coords_complemented.spherical_elevation[complement_mask]  # (N, 3): [az_rad, el_rad, r]
complement_az_deg = np.rad2deg(comp_sph_el[:, 0])
# wrap to [-180, 180]
complement_az_deg = (complement_az_deg + 180) % 360 - 180
complement_el_deg = np.rad2deg(comp_sph_el[:, 1])

# Same at the bottom pole (el=-90°): keep only az=0°
bottom_pole_mask = (complement_el_deg <= -89.9) & (np.abs(complement_az_deg) > 0.01)
complement_az_deg = complement_az_deg[~bottom_pole_mask]
complement_el_deg = complement_el_deg[~bottom_pole_mask]

print(f'Original HRTF (source grid): {len(orig_az_deg)} points')
print(f'Complement additions (mirrored source points): {len(complement_az_deg)} points')
print(f'Template grid (t-design target): {len(template_az_deg)} points')
print(f'Elevation range — Original: {orig_el_deg.min():.1f}° to {orig_el_deg.max():.1f}°, '
      f'Complement: {complement_el_deg.min():.1f}° to {complement_el_deg.max():.1f}°')

Original HRTF (source grid): 793 points
Complement additions (mirrored source points): 145 points
Template grid (t-design target): 2112 points
Elevation range — Original: -45.0° to 90.0°, Complement: -90.0° to -60.0°


In [5]:
# Summary statistics by method
print('='*80)
print('SUMMARY STATISTICS: RMSE by Method')
print('='*80)

summary_stats = all_errors_df.groupby('method')[['RMSE_mean']].agg(['mean', 'std', 'sem', 'count', 'min', 'max'])
print(summary_stats.round(3))

print('\n' + '='*80)
print('PAIRWISE COMPARISONS (RMSE_mean)')
print('='*80)

# Show mean RMSE for each method
mean_rmse = all_errors_df.groupby('method')['RMSE_mean'].mean().sort_values()
print('\nMean RMSE (sorted):')
for method, rmse in mean_rmse.items():
    print(f'  {method:15s}: {rmse:.3f} dB')

# Save results
out_csv = paths.RESULTS_DIR / 'interpolation_errors.csv'
all_errors_df.to_csv(out_csv, index=False)
print(f'\n✓ Saved to: {out_csv.relative_to(paths.ROOT)}')

SUMMARY STATISTICS: RMSE by Method
              RMSE_mean                                  
                   mean    std    sem count    min    max
method                                                   
SH                0.491  0.015  0.003    33  0.456  0.512
SHMAX             0.232  0.007  0.001    33  0.218  0.244
barumerli2023     1.489  0.037  0.006    33  1.423  1.592
barycentric       0.240  0.008  0.001    33  0.223  0.253

PAIRWISE COMPARISONS (RMSE_mean)

Mean RMSE (sorted):
  SHMAX          : 0.232 dB
  barycentric    : 0.240 dB
  SH             : 0.491 dB
  barumerli2023  : 1.489 dB

✓ Saved to: results/interpolation_errors.csv


In [6]:
latex_df = pd.DataFrame(summary_stats)
print("\n" + latex_df.to_latex(escape=False, float_format='%.2f'))


\begin{tabular}{lrrrrrr}
\toprule
 & \multicolumn{6}{r}{RMSE_mean} \\
 & mean & std & sem & count & min & max \\
method &  &  &  &  &  &  \\
\midrule
SH & 0.49 & 0.01 & 0.00 & 33 & 0.46 & 0.51 \\
SHMAX & 0.23 & 0.01 & 0.00 & 33 & 0.22 & 0.24 \\
barumerli2023 & 1.49 & 0.04 & 0.01 & 33 & 1.42 & 1.59 \\
barycentric & 0.24 & 0.01 & 0.00 & 33 & 0.22 & 0.25 \\
\bottomrule
\end{tabular}

